# Optional Practice - Components, Reciprocity, Communities, and Chess Reference

This notebook implements the optional practice tasks using the manual selections in `docs/optional_practice_instructions.md`:

- Connected components: Bitcoin Network (Part B)
- Maximal strongly connected component: Bitcoin Network (Part B)
- Reciprocal links: Bitcoin Network (Part B)
- Communities: The Wolf of Wall Street Character Network (Part A)
- Community detection method: Greedy Modularity Communities
- Chess extension: already completed in Part A

I reuse the processed Part B Bitcoin edge cache and the Part A Movie Dynamics loader approach so I do not rerun expensive preprocessing pipelines. The Bitcoin analysis uses the same sampled working graph created in Part B: directed links among the 5,000 most active endpoint IDs from the first 250,000 streamed Bitcoin rows. That means all Bitcoin results below describe the Part B working graph, not the complete raw Bitcoin archive.


## Setup and shared helpers

This cell imports the libraries, defines project paths, checks that the required cached files exist, and loads the two selected networks. The Bitcoin graph is directed because each edge has a source and destination. The movie graph is undirected because character interactions do not have a direction in the Movie Dynamics JSON.


In [ ]:
from pathlib import Path
import json
import math
import zipfile

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from networkx.readwrite import json_graph

RANDOM_SEED = 42
ROOT = Path.cwd()
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_RAW = ROOT / "data" / "raw"
FIGURES = ROOT / "exports" / "figures"
CYTOSCAPE = ROOT / "exports" / "cytoscape"
GEPHI = ROOT / "exports" / "gephi"

for folder in [FIGURES, CYTOSCAPE, GEPHI]:
    folder.mkdir(parents=True, exist_ok=True)

bitcoin_edges_cache = DATA_PROCESSED / "part_b_bitcoin_top5000_model_edges.parquet"
movie_zip_path = DATA_RAW / "movie_dynamics" / "movie-dynamics.zip"
wolf_member = "moviedynamics/2013_The_Wolf_of_Wall_Street.json"

missing = [str(path) for path in [bitcoin_edges_cache, movie_zip_path] if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required optional practice inputs:\n" + "\n".join(missing))

bitcoin_edges = pd.read_parquet(bitcoin_edges_cache).copy()
bitcoin_edges["src"] = bitcoin_edges["src"].astype(str)
bitcoin_edges["dst"] = bitcoin_edges["dst"].astype(str)

bitcoin_graph = nx.from_pandas_edgelist(
    bitcoin_edges,
    source="src",
    target="dst",
    edge_attr=["count", "mindate", "maxdate"],
    create_using=nx.DiGraph(),
)

with zipfile.ZipFile(movie_zip_path) as zf:
    with zf.open(wolf_member) as fp:
        wolf_data = json.load(fp)
wolf_graph = json_graph.node_link_graph(wolf_data)
if wolf_graph.is_directed():
    wolf_graph = wolf_graph.to_undirected()

summary = pd.DataFrame([
    {
        "network": "Bitcoin Network working graph from Part B",
        "local_input": str(bitcoin_edges_cache.relative_to(ROOT)),
        "nodes": bitcoin_graph.number_of_nodes(),
        "edges": bitcoin_graph.number_of_edges(),
        "directed": nx.is_directed(bitcoin_graph),
    },
    {
        "network": "The Wolf of Wall Street character network from Part A",
        "local_input": f"{movie_zip_path.relative_to(ROOT)}::{wolf_member}",
        "nodes": wolf_graph.number_of_nodes(),
        "edges": wolf_graph.number_of_edges(),
        "directed": nx.is_directed(wolf_graph),
    },
])
display(summary)
print(f"Bitcoin cache loaded: {bitcoin_edges_cache.relative_to(ROOT)}")
print(f"Movie graph loaded: {wolf_member}")


## 1. Connected Components Distribution - Bitcoin Network

**Requirement.** Use the selected Bitcoin network, compute and visualize the component size distribution, and explain giant-component behavior.

**Method.** The Bitcoin graph is directed, so there are two possible meanings of connected component. For this task I use **weakly connected components**: I temporarily ignore edge direction and ask which nodes are connected by any chain of transactions. This is the right view for giant-component behavior because it measures whether the sampled transaction network is mostly one connected mass, even if money does not flow both ways along every path.

**Why this method.** Strong components are stricter and are handled in the next task. Weak components give a clearer answer to the optional question: whether most nodes belong to one large component or whether the graph is fragmented into many islands.


In [ ]:
weak_components = sorted(nx.weakly_connected_components(bitcoin_graph), key=len, reverse=True)
weak_sizes = pd.Series([len(component) for component in weak_components], name="component_size")
weak_summary = pd.DataFrame({
    "number_of_weak_components": [len(weak_components)],
    "largest_component_size": [int(weak_sizes.iloc[0])],
    "total_nodes": [bitcoin_graph.number_of_nodes()],
    "largest_component_share": [float(weak_sizes.iloc[0] / bitcoin_graph.number_of_nodes())],
    "median_component_size": [float(weak_sizes.median())],
    "single_node_components": [int((weak_sizes == 1).sum())],
})

display(weak_summary)
display(weak_sizes.value_counts().sort_index().rename_axis("component_size").reset_index(name="component_count").head(20))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_sizes = weak_sizes.value_counts().sort_index().reset_index()
plot_sizes.columns = ["component_size", "component_count"]
sns.scatterplot(data=plot_sizes, x="component_size", y="component_count", ax=axes[0], s=70, color="#2f6f9f")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Bitcoin weak component size distribution")
axes[0].set_xlabel("Weak component size, log scale")
axes[0].set_ylabel("Number of components, log scale")

top_weak = pd.DataFrame({"rank": range(1, min(11, len(weak_sizes)) + 1), "component_size": weak_sizes.head(10)})
sns.barplot(data=top_weak, x="rank", y="component_size", ax=axes[1], color="#55a868")
axes[1].set_title("Largest weak components")
axes[1].set_xlabel("Component rank")
axes[1].set_ylabel("Nodes")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.0f", fontsize=8)

fig.tight_layout()
components_fig = FIGURES / "optional_bitcoin_weak_component_distribution.png"
fig.savefig(components_fig, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved figure: {components_fig.relative_to(ROOT)}")


**Interpretation.** The weak component distribution is dominated by one giant component. In this Part B working graph, the largest weak component contains almost all nodes, while the remaining components are tiny. That means the sampled Bitcoin transaction network has one main connected transaction region plus a long tail of small disconnected groups. This is common in transaction and communication networks: once direction is ignored, many accounts become connected through shared transaction chains.

### How I solved this task

I loaded the cached Part B Bitcoin edge table, built a directed NetworkX graph, computed weakly connected components with `nx.weakly_connected_components`, summarized their sizes, and saved a log-scale component-size plot. I used the cached Part B working graph to avoid rerunning the expensive archive streaming and active-node filtering.

**Limitations.** This is not the full Bitcoin network. It is the Part B sampled working graph, so the giant component share can be affected by the active-node filtering and by using only the first streamed rows from the archive. Weak components also ignore edge direction, so they do not prove that value or influence can travel both ways.


## 2. Maximal Strongly Connected Component - Bitcoin Network

**Requirement.** Use the selected Bitcoin network, extract the largest strongly connected component, visualize it, and export it if appropriate.

**Method.** A strongly connected component in a directed graph is a set of nodes where every node can reach every other node by following directed edges. I compute all strongly connected components and then select the largest one. This largest SCC is the maximal mutually reachable core of the sampled Bitcoin graph.

**Why this method.** The SCC view is stricter than the weak component view. It tells us whether directed transaction paths form a cycle-rich core or whether most links are one-way and acyclic. Since the largest SCC here is small enough to inspect, I visualize and export the whole SCC.


In [ ]:
strong_components = sorted(nx.strongly_connected_components(bitcoin_graph), key=len, reverse=True)
largest_scc_nodes = strong_components[0]
largest_scc = bitcoin_graph.subgraph(largest_scc_nodes).copy()

scc_sizes = pd.Series([len(component) for component in strong_components], name="scc_size")
scc_summary = pd.DataFrame({
    "number_of_strong_components": [len(strong_components)],
    "largest_scc_size": [largest_scc.number_of_nodes()],
    "largest_scc_edges": [largest_scc.number_of_edges()],
    "largest_scc_share_of_nodes": [largest_scc.number_of_nodes() / bitcoin_graph.number_of_nodes()],
    "single_node_sccs": [int((scc_sizes == 1).sum())],
})

display(scc_summary)

deg = dict(largest_scc.degree())
weighted = dict(largest_scc.degree(weight="count"))
for node in largest_scc.nodes:
    largest_scc.nodes[node]["degree"] = int(deg.get(node, 0))
    largest_scc.nodes[node]["weighted_degree"] = float(weighted.get(node, 0))
    largest_scc.nodes[node]["label"] = str(node)

scc_graphml = CYTOSCAPE / "optional_bitcoin_largest_scc.graphml"
scc_gexf = GEPHI / "optional_bitcoin_largest_scc.gexf"
nx.write_graphml(largest_scc, scc_graphml)
nx.write_gexf(largest_scc, scc_gexf)

fig, ax = plt.subplots(figsize=(9, 7))
pos = nx.spring_layout(largest_scc, seed=RANDOM_SEED, k=0.9)
node_sizes = [240 + 90 * deg.get(node, 0) for node in largest_scc.nodes]
edge_widths = [0.8 + math.log1p(largest_scc.edges[edge].get("count", 1)) for edge in largest_scc.edges]
nx.draw_networkx_edges(largest_scc, pos, ax=ax, alpha=0.35, arrows=True, arrowsize=10, width=edge_widths, edge_color="#666666")
nx.draw_networkx_nodes(largest_scc, pos, ax=ax, node_size=node_sizes, node_color="#4c78a8", alpha=0.9)
nx.draw_networkx_labels(largest_scc, pos, ax=ax, font_size=8)
ax.set_title("Largest strongly connected component in the sampled Bitcoin graph")
ax.axis("off")
fig.tight_layout()
scc_fig = FIGURES / "optional_bitcoin_largest_scc.png"
fig.savefig(scc_fig, dpi=160, bbox_inches="tight")
plt.show()

print(f"Saved figure: {scc_fig.relative_to(ROOT)}")
print(f"Exported GraphML: {scc_graphml.relative_to(ROOT)}")
print(f"Exported GEXF: {scc_gexf.relative_to(ROOT)}")


**Interpretation.** The largest weak component is very large, but the largest strongly connected component is much smaller. This means the graph is connected mainly through one-way transaction paths. Only a small core has full directed reachability, where each selected node can reach the others by following transaction directions.

### How I solved this task

I computed strongly connected components with `nx.strongly_connected_components`, selected the largest component, made a subgraph, added simple degree attributes, drew it with a spring layout, and exported it to GraphML and GEXF for Cytoscape or Gephi. I exported the whole largest SCC because it is small enough to handle directly.

**Limitations.** Strong reachability depends on the sampled edge set. Missing earlier/later transactions from the full archive can break directed cycles, so the SCC size should be interpreted as a property of the Part B working graph rather than the complete Bitcoin network.


## 3. Reciprocal Links - Bitcoin Network

**Requirement.** Use the selected Bitcoin network, construct a reciprocal-links subgraph, provide summary statistics, and visualize it.

**Method.** A reciprocal directed link exists when both `(u, v)` and `(v, u)` appear in the graph. I find every unordered node pair with edges in both directions and keep both directed edges in a reciprocal subgraph.

**Why this method.** Reciprocity is useful in directed transaction networks because it separates one-way interactions from pairs with repeated two-way exchange. A reciprocal subgraph can reveal small relationship structures that are hidden in the larger directed network.


In [ ]:
reciprocal_pairs = []
seen_pairs = set()
for u, v in bitcoin_graph.edges():
    pair_key = tuple(sorted([u, v]))
    if u != v and pair_key not in seen_pairs and bitcoin_graph.has_edge(v, u):
        reciprocal_pairs.append((u, v))
        seen_pairs.add(pair_key)

reciprocal_graph = nx.DiGraph()
for u, v in reciprocal_pairs:
    reciprocal_graph.add_node(u)
    reciprocal_graph.add_node(v)
    reciprocal_graph.add_edge(u, v, **bitcoin_graph.edges[u, v])
    reciprocal_graph.add_edge(v, u, **bitcoin_graph.edges[v, u])

reciprocal_stats = pd.DataFrame({
    "reciprocal_node_count": [reciprocal_graph.number_of_nodes()],
    "reciprocal_directed_edge_count": [reciprocal_graph.number_of_edges()],
    "reciprocal_pair_count": [len(reciprocal_pairs)],
    "share_of_all_directed_edges_in_reciprocal_pairs": [reciprocal_graph.number_of_edges() / bitcoin_graph.number_of_edges()],
})
display(reciprocal_stats)

reciprocal_edge_rows = []
for u, v in reciprocal_graph.edges():
    reciprocal_edge_rows.append({
        "src": u,
        "dst": v,
        "count": reciprocal_graph.edges[u, v].get("count", np.nan),
        "mindate": reciprocal_graph.edges[u, v].get("mindate", pd.NaT),
        "maxdate": reciprocal_graph.edges[u, v].get("maxdate", pd.NaT),
    })
display(pd.DataFrame(reciprocal_edge_rows).sort_values(["src", "dst"]))

for node in reciprocal_graph.nodes:
    reciprocal_graph.nodes[node]["label"] = str(node)
    reciprocal_graph.nodes[node]["degree"] = int(reciprocal_graph.degree(node))

reciprocal_graphml = CYTOSCAPE / "optional_bitcoin_reciprocal_links.graphml"
reciprocal_gexf = GEPHI / "optional_bitcoin_reciprocal_links.gexf"
nx.write_graphml(reciprocal_graph, reciprocal_graphml)
nx.write_gexf(reciprocal_graph, reciprocal_gexf)

fig, ax = plt.subplots(figsize=(8, 6))
if reciprocal_graph.number_of_nodes() > 0:
    pos = nx.spring_layout(reciprocal_graph, seed=RANDOM_SEED, k=1.2)
    widths = [1.0 + math.log1p(reciprocal_graph.edges[edge].get("count", 1)) for edge in reciprocal_graph.edges]
    nx.draw_networkx_edges(reciprocal_graph, pos, ax=ax, arrows=True, arrowsize=14, width=widths, alpha=0.55, edge_color="#444444")
    nx.draw_networkx_nodes(reciprocal_graph, pos, ax=ax, node_size=900, node_color="#dd8452", alpha=0.9)
    nx.draw_networkx_labels(reciprocal_graph, pos, ax=ax, font_size=9)
else:
    ax.text(0.5, 0.5, "No reciprocal links found", ha="center", va="center")
ax.set_title("Reciprocal directed links in the sampled Bitcoin graph")
ax.axis("off")
fig.tight_layout()
reciprocal_fig = FIGURES / "optional_bitcoin_reciprocal_links.png"
fig.savefig(reciprocal_fig, dpi=160, bbox_inches="tight")
plt.show()

print(f"Saved figure: {reciprocal_fig.relative_to(ROOT)}")
print(f"Exported GraphML: {reciprocal_graphml.relative_to(ROOT)}")
print(f"Exported GEXF: {reciprocal_gexf.relative_to(ROOT)}")


**Interpretation.** The reciprocal subgraph is very small compared with the full sampled Bitcoin graph. Most directed Bitcoin links in this working graph do not have a matching reverse edge, which is consistent with a transaction network where many source-destination relationships are one-way in the sampled period.

### How I solved this task

I checked every directed Bitcoin edge for the reverse edge, kept each reciprocal pair once, then added both directions to a new directed subgraph. I displayed summary statistics, listed the reciprocal edges, drew the subgraph, and exported it for external graph tools.

**Limitations.** A missing reverse edge in the sample does not prove the pair never transacted in both directions in the full dataset. It only means the reverse direction is absent from the Part B working graph after sampling and active-node filtering.


## 4. Communities - The Wolf of Wall Street Character Network

**Requirement.** Use the selected Wolf of Wall Street character network, apply **Greedy Modularity Communities exactly**, identify the second most central vertex in every detected community, and explain why the result is meaningful.

**Method.** Greedy modularity communities start with each node in its own community and repeatedly merge communities when the merge improves modularity. Modularity is high when there are many edges inside communities and fewer edges between communities than expected by chance. After detecting communities, I compute weighted PageRank inside each community subgraph and select the second-ranked character.

**Why this method.** The assignment selected Greedy Modularity Communities exactly. It is appropriate here because the movie graph is small, undirected, and dense enough for community detection. The second most central character is useful because it avoids simply naming the obvious lead character in each group and instead highlights another important character in that local story cluster.


In [ ]:
communities = list(nx.algorithms.community.greedy_modularity_communities(wolf_graph, weight="weight"))
communities = sorted(communities, key=len, reverse=True)

community_records = []
node_to_community = {}
for community_id, nodes in enumerate(communities, start=1):
    subgraph = wolf_graph.subgraph(nodes).copy()
    if subgraph.number_of_nodes() == 1:
        centrality = {next(iter(subgraph.nodes)): 1.0}
    else:
        centrality = nx.pagerank(subgraph, weight="weight")
    ranked = sorted(centrality.items(), key=lambda item: item[1], reverse=True)
    first = ranked[0]
    second = ranked[1] if len(ranked) > 1 else (None, np.nan)
    internal_edges = subgraph.number_of_edges()
    possible_edges = subgraph.number_of_nodes() * (subgraph.number_of_nodes() - 1) / 2
    density = internal_edges / possible_edges if possible_edges else 0.0
    community_records.append({
        "community_id": community_id,
        "size": subgraph.number_of_nodes(),
        "internal_edges": internal_edges,
        "density": density,
        "most_central_vertex": first[0],
        "most_central_pagerank": first[1],
        "second_most_central_vertex": second[0],
        "second_most_central_pagerank": second[1],
    })
    for node in nodes:
        node_to_community[node] = community_id

community_df = pd.DataFrame(community_records)
display(community_df)

modularity = nx.algorithms.community.modularity(wolf_graph, communities, weight="weight")
print(f"Detected {len(communities)} communities with weighted modularity {modularity:.3f}")

community_graph = wolf_graph.copy()
weighted_degree = dict(community_graph.degree(weight="weight"))
for node in community_graph.nodes:
    community_graph.nodes[node]["community"] = int(node_to_community[node])
    community_graph.nodes[node]["weighted_degree"] = float(weighted_degree.get(node, 0))
    community_graph.nodes[node]["label"] = str(node)

community_graphml = CYTOSCAPE / "optional_wolf_greedy_modularity_communities.graphml"
community_gexf = GEPHI / "optional_wolf_greedy_modularity_communities.gexf"
nx.write_graphml(community_graph, community_graphml)
nx.write_gexf(community_graph, community_gexf)

palette = sns.color_palette("tab10", n_colors=max(3, len(communities)))
node_colors = [palette[node_to_community[node] - 1] for node in wolf_graph.nodes]
node_sizes = [140 + 7 * weighted_degree.get(node, 0) for node in wolf_graph.nodes]
pos = nx.spring_layout(wolf_graph, seed=RANDOM_SEED, weight="weight", k=0.55)

fig, ax = plt.subplots(figsize=(12, 9))
edge_widths = [0.35 + 0.04 * math.log1p(wolf_graph.edges[edge].get("weight", 1)) for edge in wolf_graph.edges]
nx.draw_networkx_edges(wolf_graph, pos, ax=ax, width=edge_widths, alpha=0.22, edge_color="#555555")
nx.draw_networkx_nodes(wolf_graph, pos, ax=ax, node_size=node_sizes, node_color=node_colors, alpha=0.90)

labels = {}
important_nodes = set(community_df["second_most_central_vertex"].dropna()) | {"Jordan Belfort"}
for node in wolf_graph.nodes:
    if node in important_nodes:
        labels[node] = node
nx.draw_networkx_labels(wolf_graph, pos, labels=labels, ax=ax, font_size=8)

ax.set_title("Greedy modularity communities in The Wolf of Wall Street character network")
ax.axis("off")
fig.tight_layout()
community_fig = FIGURES / "optional_wolf_greedy_modularity_communities.png"
fig.savefig(community_fig, dpi=160, bbox_inches="tight")
plt.show()

print(f"Saved figure: {community_fig.relative_to(ROOT)}")
print(f"Exported GraphML: {community_graphml.relative_to(ROOT)}")
print(f"Exported GEXF: {community_gexf.relative_to(ROOT)}")


**Interpretation.** Greedy modularity separates the movie network into groups of characters that interact more heavily with one another than with the rest of the cast. The second most central vertex in each community is meaningful because it points to an important supporting character within that local group. For example, if the main character dominates one community, the second-ranked character helps explain who else anchors that part of the interaction structure.

### How I solved this task

I loaded the same Wolf of Wall Street graph used in Part A, ran `nx.algorithms.community.greedy_modularity_communities` with edge weights, calculated PageRank inside each detected community, and selected the second-highest PageRank node in each community. I then saved a community-colored visualization and exported the graph with a `community` node attribute.

**Limitations.** Greedy modularity is a structural algorithm, not a story-aware method. It only sees interaction edges and weights, so the detected communities may reflect scene co-appearance patterns rather than formal plot groups. Very small communities also make the idea of a second most central vertex less informative.


## 5. Chess Extension Reference

**Requirement.** Do not duplicate the Part A chess analysis. Briefly explain that the chess extension was already completed in Part A and reference that work.

**Explanation.** The manual selection says `DO_CHESS_EXTENSION_HERE_OR_IN_PART_A = PART_A`. I therefore do not rerun the large FICS chess analysis in this optional notebook. Part A already handled the large-graph safety requirement by streaming the first 200,000 chess interaction rows from `data/raw/chess/fics.tar.gz`, caching the sample at `data/processed/fics_interactions_first_200000.parquet`, identifying the top sampled players by interaction count, visualizing a small neighborhood subgraph, and exporting GraphML/GEXF files.

The relevant Part A outputs are:

- Figure: `exports/figures/a6_fics_sampled_top_players.png`
- Cytoscape export: `exports/cytoscape/fics_sampled_top_players.graphml`
- Gephi export: `exports/gephi/fics_sampled_top_players.gexf`

### How I solved this task

I followed the manual selection and treated chess as already completed in Part A. This avoids duplicating an expensive large-graph workflow and keeps this notebook focused on the optional Bitcoin and movie-network tasks.

**Limitations.** This section does not add new chess results. It only documents where the completed Part A chess extension can be found.


## Optional Practice Output Summary

This final cell records the generated figures and exports so the grader can find the outputs quickly.


In [ ]:
created_outputs = [
    {"type": "figure", "path": FIGURES / "optional_bitcoin_weak_component_distribution.png"},
    {"type": "figure", "path": FIGURES / "optional_bitcoin_largest_scc.png"},
    {"type": "figure", "path": FIGURES / "optional_bitcoin_reciprocal_links.png"},
    {"type": "figure", "path": FIGURES / "optional_wolf_greedy_modularity_communities.png"},
    {"type": "cytoscape_export", "path": CYTOSCAPE / "optional_bitcoin_largest_scc.graphml"},
    {"type": "cytoscape_export", "path": CYTOSCAPE / "optional_bitcoin_reciprocal_links.graphml"},
    {"type": "cytoscape_export", "path": CYTOSCAPE / "optional_wolf_greedy_modularity_communities.graphml"},
    {"type": "gephi_export", "path": GEPHI / "optional_bitcoin_largest_scc.gexf"},
    {"type": "gephi_export", "path": GEPHI / "optional_bitcoin_reciprocal_links.gexf"},
    {"type": "gephi_export", "path": GEPHI / "optional_wolf_greedy_modularity_communities.gexf"},
]
output_df = pd.DataFrame([
    {"type": item["type"], "path": str(item["path"].relative_to(ROOT)), "exists": item["path"].exists()}
    for item in created_outputs
])
display(output_df)

optional_summary = {
    "bitcoin_input_cache": str(bitcoin_edges_cache.relative_to(ROOT)),
    "bitcoin_nodes": bitcoin_graph.number_of_nodes(),
    "bitcoin_edges": bitcoin_graph.number_of_edges(),
    "weak_components": len(weak_components),
    "largest_weak_component_size": int(weak_sizes.iloc[0]),
    "largest_scc_size": largest_scc.number_of_nodes(),
    "reciprocal_pair_count": len(reciprocal_pairs),
    "wolf_graph_member": wolf_member,
    "wolf_communities": len(communities),
    "wolf_modularity": float(modularity),
    "outputs": output_df.to_dict(orient="records"),
}
summary_path = DATA_PROCESSED / "optional_practice_summary.json"
summary_path.write_text(json.dumps(optional_summary, indent=2))
print(f"Summary written to {summary_path.relative_to(ROOT)}")
